# Fanfics Dataset Analysis using Apache Spark

## Overview
- This notebook demonstrates the setup and verification of a Spark environment on the SDSC Expanse cluster,
- followed by loading and validating a large-scale fanfiction dataset.

## Objectives
- Configure SparkSession based on allocated cluster resources
- Load dataset stored on Lustre filesystem
- Verify distributed processing
- Prepare for large-scale data analysis

## Dataset
- Source: https://huggingface.co/datasets/marianna13/fanfics

## Spark Configuration

The SparkSession is configured for the allocated resources in the Expanse Jupyter environment, with two adjustments specific to this corpus.

### Allocated Resources
- Total Cores: 16
- Total Memory: 128 GB

### Configuration Formula (per `SPARK_HPC_BEST_PRACTICES.md`)
- Driver Memory: **16 GB** (see "Local-mode override" below)
- Executor Instances: Total Cores - 1 = 15
- Executor Memory: (Total Memory - Driver Memory) / Executor Instances = (128 - 16) / 15 ≈ 7 GB → kept at 8 GB for documentation; not active in local mode

### Local-mode override
Expanse JupyterLab launches Spark in `local[*]` mode (no YARN provisioning), so the driver JVM IS the executor — all task memory pressure falls on the driver. We bump driver memory to **16 GB** to give 16 concurrent tasks enough headroom. `executor.memory` and `executor.instances` are kept in the builder per the documented formula but are inactive in local mode.

### Vectorized Parquet reader disabled
The corpus contains `TEXT` records up to ~6 MB. Spark's default vectorized Parquet reader allocates contiguous buffers proportional to its batch size (4,096 rows) and column size, which OOMs the JVM when batches contain large TEXT values. Disabling vectorized reading (`spark.sql.parquet.enableVectorizedReader=false`) switches to row-by-row reads with bounded per-row memory at a small read-throughput cost — acceptable for exploratory analysis.

In [1]:
import requests
import pandas as pd

In [2]:
import os
from pyspark.sql import SparkSession

# Redirect Spark shuffle/spill scratch from /tmp (small, on compute node) to
# Lustre (TB-scale, per-user project space). This is necessary because the
# corpus has multi-MB TEXT rows; any shuffle on TEXT (e.g., dropDuplicates)
# spills enough data to exhaust /tmp on the compute node.
user = os.environ.get('USER', 'unknown')
spark_local_dir = f"/expanse/lustre/projects/uci157/{user}/spark-scratch"
os.makedirs(spark_local_dir, exist_ok=True)

spark = SparkSession.builder \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.instances", 15) \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.local.dir", spark_local_dir) \
    .getOrCreate()

# Display Spark session to confirm initialization
spark

In [3]:
conf = spark.sparkContext.getConf().getAll()

for key, value in conf:
    print(f"{key}: {value}")

spark.executor.instances: 15
spark.executor.id: driver
spark.driver.memory: 16g
spark.app.name: pyspark-shell
spark.driver.extraJavaOptions: -Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false
spark.local.dir: /expanse/lustre/projects/uci157/dpham

In [4]:
print("Default Parallelism:", spark.sparkContext.defaultParallelism)

Default Parallelism: 16


In [5]:
import os
import glob

# ----------------------------------------
# Load Dataset from Lustre Filesystem
# ----------------------------------------
# Each member has his own 186 GB corpus copy at <repo>/shared/fanfics/.
# Resolve the path portably so either member's run finds his own copy.

candidates = [
    os.path.join(os.path.dirname(os.getcwd()), "shared", "fanfics"),  # repo-relative when cwd=notebook/
    os.path.join(os.getcwd(), "shared", "fanfics"),                    # repo-relative when cwd=repo root
    "/expanse/lustre/projects/uci157/dpham5/fanfic-spark-analysis/shared/fanfics",  # Derek absolute
    "/expanse/lustre/projects/uci157/mhayeri/fanfics-analysis/shared/fanfics",       # Mustafa absolute
]
data_path = next((p for p in candidates if os.path.exists(p)), None)
if data_path is None:
    raise FileNotFoundError(f"None of the candidate corpus paths exist: {candidates}")
print(f"Using data path: {data_path}")

# ----------------------------------------
# Schema heterogeneity across Parquet shards
# ----------------------------------------
# The marianna13/fanfics corpus was produced in heterogeneous batches:
#   - `language` is encoded as string in most shards but as INT32 in some
#   - `__null_dask_index__` is present in some shards but not others
# Spark's default read picks one global schema and fails on mismatches:
#
#   Py4JJavaError: Parquet column cannot be converted in file
#       part-03500-...snappy.parquet. Column: [language],
#       Expected: string, Found: INT32.
#
# Workaround: read each shard's footer, split the file list by `language`
# physical type, load the two subsets, cast the INT32 subset to string, and
# union with allowMissingColumns=True so column-set differences become NULL.
# This preserves all rows; the heterogeneity is itself a finding documented
# in §3 frequency tables and addressed in the §5 preprocessing plan.

from pyspark.sql.functions import col

all_files = sorted(glob.glob(f"{data_path}/*.parquet"))
print(f"Total shards on disk: {len(all_files)}")

# Use pyarrow for fast metadata reads if available; fall back to Spark otherwise.
try:
    import pyarrow.parquet as pq
    def _lang_type(f):
        return str(pq.read_metadata(f).schema.to_arrow_schema().field('language').type)
except ImportError:
    def _lang_type(f):
        return dict(spark.read.parquet(f).dtypes).get('language', 'MISSING')

str_files = []
int_files = []
for i, f in enumerate(all_files):
    t = _lang_type(f)
    if t == 'string':
        str_files.append(f)
    else:
        int_files.append(f)
    if (i + 1) % 200 == 0:
        print(f"  Scanned {i+1}/{len(all_files)} schemas...")

print(f"String-language shards: {len(str_files)}")
print(f"Int-language shards:    {len(int_files)}")

parts = []
if str_files:
    parts.append(spark.read.parquet(*str_files))
if int_files:
    parts.append(
        spark.read.parquet(*int_files).withColumn("language", col("language").cast("string"))
    )

df = parts[0]
for p in parts[1:]:
    df = df.unionByName(p, allowMissingColumns=True)

print("Dataset successfully loaded with normalized language schema.")

Using data path: /expanse/lustre/projects/uci157/dpham5/fanfic-spark-analysis/shared/fanfics
Total shards on disk: 2018
  Scanned 200/2018 schemas...
  Scanned 400/2018 schemas...
  Scanned 600/2018 schemas...
  Scanned 800/2018 schemas...
  Scanned 1000/2018 schemas...
  Scanned 1200/2018 schemas...
  Scanned 1400/2018 schemas...
  Scanned 1600/2018 schemas...
  Scanned 1800/2018 schemas...
  Scanned 2000/2018 schemas...
String-language shards: 2016
Int-language shards:    2
Dataset successfully loaded with normalized language schema.


In [6]:
# ----------------------------------------
# Trigger Spark Job to Validate Data Loading
# ----------------------------------------

row_count = df.count()
print(f"Total number of observations: {row_count}")

Total number of observations: 5252058


In [7]:
# ----------------------------------------
# Inspect Dataset Schema
# ----------------------------------------

df.printSchema()

root
 |-- __null_dask_index__: long (nullable = true)
 |-- TEXT: string (nullable = true)
 |-- CATEGORY: string (nullable = true)
 |-- SOURCE: string (nullable = true)
 |-- language: string (nullable = true)
 |-- text_len: long (nullable = true)
 |-- perplexity_score: double (nullable = true)



In [8]:
# ----------------------------------------
# Display Sample Rows
# ----------------------------------------

# df.show(5, truncate=False)

In [9]:
# ----------------------------------------
# Check Number of Partitions
# ----------------------------------------

num_partitions = df.rdd.getNumPartitions()

print("Number of partitions:", num_partitions)

Number of partitions: 1945


## Section 3 — Data Exploration

All exploration uses Spark DataFrames per the assignment requirement. Plots in §4 use sampled or aggregated results converted to Pandas after the Spark stages.

### 3a. Total observation count

`df.count()` is shown above (5,252,058 rows).

### 3b. Column descriptions and distributions

Numeric columns (`text_len`, `perplexity_score`) get `describe()`. Categorical columns (`CATEGORY`, `SOURCE`, `language`) get frequency tables and distinct counts. The `__null_dask_index__` column is a Dask leftover and will be dropped in preprocessing.

In [10]:
df.describe(["text_len", "perplexity_score"]).show()

+-------+------------------+------------------+
|summary|          text_len|  perplexity_score|
+-------+------------------+------------------+
|  count|           5252058|           5252058|
|   mean| 59265.66334377876|12093.941925317644|
| stddev|147270.26689221288|2866.4860636911285|
|    min|              5001|              56.8|
|    max|          93626685|           69992.0|
+-------+------------------+------------------+



In [11]:
from pyspark.sql import functions as F

df.groupBy('language').count().orderBy(F.desc('count')).show(30, truncate=False)
print("Distinct languages:", df.select('language').distinct().count())

+--------+-------+
|language|count  |
+--------+-------+
|en      |4591384|
|es      |292400 |
|fr      |137479 |
|id      |121040 |
|pt      |58073  |
|de      |24970  |
|pl      |7454   |
|NULL    |3088   |
|da      |2990   |
|it      |2264   |
|ru      |2045   |
|hu      |2008   |
|nl      |1572   |
|sv      |1171   |
|cs      |1057   |
|fi      |972    |
|tl      |692    |
|ca      |387    |
|sk      |267    |
|ro      |117    |
|no      |97     |
|tr      |91     |
|he      |73     |
|hr      |57     |
|el      |57     |
|bg      |54     |
|ar      |31     |
|so      |22     |
|sw      |20     |
|af      |18     |
+--------+-------+
only showing top 30 rows

Distinct languages: 45


In [12]:
df.groupBy('SOURCE').count().orderBy(F.desc('count')).show(10, truncate=False)
print("Distinct SOURCE values:", df.select('SOURCE').distinct().count())

+----------+-------+
|SOURCE    |count  |
+----------+-------+
|Fanfiction|5252058|
+----------+-------+

Distinct SOURCE values: 1


In [13]:
df.groupBy('CATEGORY').count().orderBy(F.desc('count')).show(50, truncate=False)
print("Approx distinct CATEGORY values:",
      df.agg(F.approx_count_distinct('CATEGORY')).collect()[0][0])

+-------------------------------------+------+
|CATEGORY                             |count |
+-------------------------------------+------+
|                                     |242546|
|Harry Potter, Romance                |83528 |
|Harry Potter, Humor, Romance         |66201 |
|Harry Potter, Drama, Romance         |58962 |
|Harry Potter                         |47382 |
|Naruto, Romance                      |44502 |
|Naruto, Humor, Romance               |44427 |
|Naruto, Drama, Romance               |34016 |
|Harry Potter, Angst, Romance         |31134 |
|Harry Potter, Adventure, Romance     |23142 |
|Naruto, Adventure, Romance           |21268 |
|Harry Potter, Humor                  |20818 |
|Naruto                               |18820 |
|Harry Potter, Hurt-Comfort, Romance  |15748 |
|Harry Potter, Friendship, Romance    |15622 |
|Naruto, Hurt-Comfort, Romance        |15057 |
|Glee, Romance                        |14258 |
|Naruto, Angst, Romance               |13358 |
|Hetalia - Ax

In [14]:
quantiles_text_len = df.approxQuantile("text_len",
    [0.0, 0.01, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0], 0.001)
quantiles_perp = df.approxQuantile("perplexity_score",
    [0.0, 0.01, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0], 0.001)

print("text_len quantiles:        ", quantiles_text_len)
print("perplexity_score quantiles:", quantiles_perp)

text_len quantiles:         [5001.0, 5150.0, 9596.0, 19209.0, 51525.0, 135217.0, 588906.0, 93626685.0]
perplexity_score quantiles: [56.8, 8693.9, 10563.5, 11563.8, 12894.8, 14576.2, 21649.8, 69992.0]


### 3c. Missing and duplicate values

Nulls are counted per column with a single aggregation. Duplicates are defined by exact `TEXT` equality — the practical definition for a text corpus where the same prose may appear under multiple `CATEGORY` rows. To keep the shuffle tractable on multi-MB TEXT values, we hash each TEXT to a 32-byte md5 digest first and count distinct hashes; collisions at this scale are astronomically unlikely (~2^-128).

In [15]:
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show(truncate=False)

+-------------------+----+--------+------+--------+--------+----------------+
|__null_dask_index__|TEXT|CATEGORY|SOURCE|language|text_len|perplexity_score|
+-------------------+----+--------+------+--------+--------+----------------+
|5119306            |0   |0       |0     |3088    |0       |0               |
+-------------------+----+--------+------+--------+--------+----------------+



In [16]:
# Duplicate detection via md5 hash of TEXT.
# A direct dropDuplicates(['TEXT']) shuffles full TEXT values (up to ~6 MB
# each) and exhausts local scratch on the compute node. Hashing TEXT to a
# 32-byte digest first keeps the shuffle tractable; md5 collisions at this
# scale are astronomically unlikely (~2^-128).
total = df.count()
unique_texts = df.select(F.md5('TEXT').alias('text_hash')).distinct().count()
print(f"Total rows:           {total:,}")
print(f"Unique TEXT rows:     {unique_texts:,}")
print(f"Duplicate TEXT rows:  {total - unique_texts:,}")

Total rows:           5,252,058
Unique TEXT rows:     4,914,503
Duplicate TEXT rows:  337,555


## Spark UI Verification

A Spark UI screenshot is included showing:
- Multiple executors active
- Tasks distributed across nodes

This confirms that the dataset is processed in a distributed manner.



In [17]:
sc = spark.sparkContext

url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

response = requests.get(url)
executors = response.json()

# Convert to DataFrame
exec_df = pd.DataFrame(executors)[
    ['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']
]

# Convert memory to GB
exec_df['maxMemory_GB'] = (exec_df['maxMemory'] / (1024**3)).round(2)

exec_df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,16,10119177830,0,True,9.42


## Section 4 — Data Plots

Plotting workflow: Spark aggregation/sampling → `.toPandas()` → matplotlib. Sampling uses a small fraction (~0.1%) for histograms and scatter plots; bar charts use full aggregated counts. Each plot is followed by a written interpretation.

Notes from §3 that drive the plot choices:
- 45 distinct language tags (heavy English dominance)
- `text_len` ranges from 5,001 to ~93 million characters per record (single-record outliers up to ~93 MB)
- `perplexity_score` ranges from ~57 to ~70,000
- 278,388 distinct `CATEGORY` values; the top value is the empty string (242,546 rows) — a known data-quality artifact handled in §5

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import functions as F

plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Plot 1: Rows per language (log-scale bar over all distinct languages)
lang_counts = (
    df.groupBy('language')
      .count()
      .orderBy(F.desc('count'))
      .toPandas()
)

plt.figure(figsize=(14, 5))
plt.bar(lang_counts['language'].astype(str), lang_counts['count'])
plt.yscale('log')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.ylabel('row count (log scale)')
plt.title(f'Rows per language ({len(lang_counts)} distinct values, log scale)')
plt.tight_layout()
plt.show()

print(f'Distinct languages: {len(lang_counts)}')
print(f'Top 5 languages by row count:')
print(lang_counts.head(5).to_string(index=False))

**Plot 1 — Rows per language.** A log-scale bar chart over all 45 distinct language tags reveals the magnitude of language imbalance in the corpus. The dominant language (English) has many orders of magnitude more rows than the smallest tier. This imbalance directly informs the §5 preprocessing plan: any cross-lingual transfer evaluation must account for low-resource languages, and per-language stratified sampling needs to handle the long tail. The 45-distinct count is higher than the abstract's claim of 22 — this was surfaced in §3 and the README has been corrected accordingly.

In [ ]:
# Plot 2: text_len distribution — linear and log10 views (0.1% sample, ~5K rows)
sample_text_len = df.sample(fraction=0.001, seed=42).select('text_len').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sample_text_len['text_len'], bins=100)
axes[0].set_yscale('log')
axes[0].set_xlabel('text_len (characters)')
axes[0].set_ylabel('count (log scale)')
axes[0].set_title(f'text_len distribution (linear x, log y; n={len(sample_text_len):,})')

axes[1].hist(np.log10(sample_text_len['text_len'].clip(lower=1)), bins=100)
axes[1].set_xlabel('log10(text_len)')
axes[1].set_ylabel('count')
axes[1].set_title('log10(text_len) distribution')

plt.tight_layout()
plt.show()

print('text_len sample summary:')
print(sample_text_len['text_len'].describe().round(2).to_string())

**Plot 2 — `text_len` distribution.** The histogram is heavily right-skewed: most records cluster in the few-thousand-character range, with a long tail extending out to multi-million-character outliers. The linear-x view is unreadable on its own, so a log10-x version is shown alongside — that view reveals an approximately log-normal-shaped body. From §3 quantiles, the maximum single-record `TEXT` reaches ~93 MB (~93 million characters), much larger than the abstract's assumed ~6 MB. These mega-records drive the §5 preprocessing decisions: cap or filter `text_len` at the 99th percentile to prevent partition skew during downstream model training, and the disabled vectorized Parquet reader is retained because of these outliers.

In [ ]:
# Plot 3: perplexity_score distribution (0.1% sample)
sample_perp = df.sample(fraction=0.001, seed=42).select('perplexity_score').toPandas()
sample_perp = sample_perp['perplexity_score'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sample_perp, bins=100)
axes[0].set_xlabel('perplexity_score')
axes[0].set_ylabel('count')
axes[0].set_title(f'perplexity_score distribution (linear; n={len(sample_perp):,})')

axes[1].hist(np.log10(sample_perp.clip(lower=1)), bins=100)
axes[1].set_xlabel('log10(perplexity_score)')
axes[1].set_ylabel('count')
axes[1].set_title('log10(perplexity_score) distribution')

plt.tight_layout()
plt.show()

print('perplexity_score sample summary:')
print(sample_perp.describe().round(2).to_string())

**Plot 3 — `perplexity_score` distribution.** Perplexity measures how predictable each record is under a baseline language model. The linear-x view is dominated by a few extreme high-perplexity outliers (the §3 max was ~70,000), so a log10-x version is shown alongside — that view reveals the bulk of the corpus sits in a much narrower predictability band. The shape of this distribution informs the §5 quality-filter threshold: extreme high-perplexity records (gibberish, broken encodings, non-natural-language artifacts) can be excluded; extreme low-perplexity records (boilerplate, copy-paste artifacts) likely contribute little useful signal.

In [ ]:
# Plot 4: Top-25 categories by row count (horizontal bar)
top_cats = (
    df.groupBy('CATEGORY')
      .count()
      .orderBy(F.desc('count'))
      .limit(25)
      .toPandas()
)

# Replace the empty-string CATEGORY label with a visible placeholder for plotting
top_cats['CATEGORY_label'] = top_cats['CATEGORY'].replace('', '<empty string>')

plt.figure(figsize=(12, 8))
plt.barh(top_cats['CATEGORY_label'][::-1], top_cats['count'][::-1])
plt.xlabel('row count')
plt.title('Top 25 categories by row count')
plt.tight_layout()
plt.show()

print(f'Total distinct CATEGORY values (from §3): 278,388')
print(f'Top 5 categories shown:')
print(top_cats[['CATEGORY_label', 'count']].head(5).to_string(index=False))

**Plot 4 — Top-25 categories.** The target column is heavily long-tailed: of 278,388 distinct `CATEGORY` values, the top entry is the empty string (242,546 rows) — a data-quality artifact, not a real category. Beyond that, the named fandoms (Harry Potter, Naruto, etc.) dominate by 1-2 orders of magnitude over the rest of the long tail. For Milestone 3 we will (a) drop empty-`CATEGORY` rows entirely, then (b) restrict the classifier to the top-K most frequent named categories (K ≈ 50, refined from this plot) and aggregate the remaining tail into an "other" class — keeping the problem tractable and class balance manageable.

In [ ]:
# Plot 5: Mean ± stddev of text_len for the top-5 languages by row count
top5_langs = [
    r['language']
    for r in (
        df.groupBy('language').count().orderBy(F.desc('count')).limit(5).collect()
    )
]

per_lang_stats = (
    df.filter(F.col('language').isin(top5_langs))
      .groupBy('language')
      .agg(
          F.avg('text_len').alias('mean_text_len'),
          F.stddev('text_len').alias('std_text_len'),
          F.expr('percentile_approx(text_len, 0.5)').alias('median_text_len'),
      )
      .toPandas()
)

# Preserve top-5 ordering by row count
per_lang_stats = per_lang_stats.set_index('language').loc[top5_langs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(per_lang_stats.index.astype(str), per_lang_stats['mean_text_len'],
       yerr=per_lang_stats['std_text_len'], capsize=5, alpha=0.8, label='mean ± stddev')
ax.scatter(per_lang_stats.index.astype(str), per_lang_stats['median_text_len'],
           color='red', zorder=3, label='median')
ax.set_ylabel('text_len (characters)')
ax.set_title('Mean ± stddev and median text_len for top-5 languages')
ax.legend()
plt.tight_layout()
plt.show()

print('Top-5 language text_len statistics:')
print(per_lang_stats.round(0).to_string())

**Plot 5 — `text_len` by top-5 languages.** The bar shows mean `text_len` with stddev error bars; the red dots show the median. Differences between mean and median highlight the right-skew in each language's record-length distribution (a few mega-records pull means well above medians). Differences across languages hint at writing-style or platform-effect variation between language communities, and matter for the cross-lingual transfer plan in §5: a model trained on long English texts may generalize poorly to a community that writes shorter pieces, and outlier-driven means need to be handled with median-based filtering rather than mean-based filtering.

In [ ]:
# Plot 6: text_len vs perplexity_score scatter (0.1% sample, log-x)
scatter_sample = (
    df.sample(fraction=0.001, seed=42)
      .select('text_len', 'perplexity_score')
      .dropna()
      .toPandas()
)

plt.figure(figsize=(9, 6))
plt.scatter(scatter_sample['text_len'], scatter_sample['perplexity_score'],
            alpha=0.2, s=5)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('text_len (log scale)')
plt.ylabel('perplexity_score (log scale)')
plt.title(f'text_len vs perplexity_score (0.1% sample, n={len(scatter_sample):,})')
plt.tight_layout()
plt.show()

# Pearson correlation on log-log values for a quick redundancy check
corr = (
    np.log10(scatter_sample['text_len'].clip(lower=1))
       .corr(np.log10(scatter_sample['perplexity_score'].clip(lower=1)))
)
print(f'Pearson correlation (log10 text_len, log10 perplexity_score): {corr:.3f}')

**Plot 6 — `text_len` vs `perplexity_score`.** Scatter (log-x for `text_len`, log-y for `perplexity_score`) reveals whether longer texts are systematically more or less predictable. Low correlation suggests `text_len` and `perplexity_score` carry distinct signal — both worth keeping as features. Strong correlation would indicate redundancy and let us drop one in §5 preprocessing. The printed Pearson correlation on log-log values quantifies this; in practice we expect a weak negative correlation (longer texts tend to be slightly more predictable for a baseline LM), but small enough that both features stay.